In [1]:
import pandas as pd
import numpy as np

print("1. Loading processed data...")
df = pd.read_csv('../data/processed/downsampled_sales.csv')
df['date'] = pd.to_datetime(df['date'])

# Sort strictly by item and date to ensure temporal order
df = df.sort_values(['item_id', 'date']).reset_index(drop=True)

print("2. Extracting Date Features...")
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print("3. Creating Lag Features (Looking into the past)...")
# Lag 7: What were the sales exactly 7 days ago? (Used for Seasonal Naive)
df['sales_lag_7'] = df.groupby('item_id')['sales'].shift(7)

# Lag 28: What were the sales 28 days ago? (Captures monthly cycles)
df['sales_lag_28'] = df.groupby('item_id')['sales'].shift(28)

# Rolling Mean: What was the average daily sale over the last 7 days?
# We must shift by 1 first so we don't include today's sales (which would be cheating/data leakage)
df['rolling_mean_7'] = df.groupby('item_id')['sales'].transform(
    lambda x: x.shift(1).rolling(window=7).mean()
)

print("4. Dropping initial rows with NaN values due to lags...")
df_features = df.dropna().reset_index(drop=True)

print(f"Feature engineering complete. Rows remaining: {len(df_features)}")
display(df_features[['date', 'item_id', 'sales', 'sales_lag_7', 'rolling_mean_7']].head(10))

1. Loading processed data...
2. Extracting Date Features...
3. Creating Lag Features (Looking into the past)...
4. Dropping initial rows with NaN values due to lags...
Feature engineering complete. Rows remaining: 15100


,date,item_id,sales,sales_lag_7,rolling_mean_7
0,2011-03-09,HOBBIES_1_001,0,0.0,0.0
1,2011-03-16,HOBBIES_1_001,0,0.0,0.0
2,2011-03-17,HOBBIES_1_001,0,0.0,0.0
3,2011-03-20,HOBBIES_1_001,0,0.0,0.0
4,2011-04-24,HOBBIES_1_001,0,0.0,0.0
5,2011-04-26,HOBBIES_1_001,0,0.0,0.0
6,2011-05-05,HOBBIES_1_001,0,0.0,0.0
7,2011-05-08,HOBBIES_1_001,0,0.0,0.0
8,2011-05-30,HOBBIES_1_001,0,0.0,0.0
9,2011-05-31,HOBBIES_1_001,0,0.0,0.0


In [2]:
print("1. Chronological Train-Test Split...")
# We will use the last 30 days of data as our unseen Test Set
max_date = df_features['date'].max()
test_start_date = max_date - pd.Timedelta(days=30)

train_data = df_features[df_features['date'] < test_start_date]
test_data = df_features[df_features['date'] >= test_start_date]

print(f"Training on dates: {train_data['date'].min().date()} to {train_data['date'].max().date()}")
print(f"Testing on dates: {test_data['date'].min().date()} to {test_data['date'].max().date()}")

1. Chronological Train-Test Split...
Training on dates: 2011-03-09 to 2016-02-17
Testing on dates: 2016-03-17 to 2016-03-27


In [3]:
def calculate_wmape(y_true, y_pred):
    # WMAPE = Sum of absolute errors / Sum of actual values
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

def calculate_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

print("1. Generating Seasonal Naïve Predictions (Predicting 'Lag 7' for today)...")
# Our baseline simply guesses that today's sales = sales from exactly one week ago
y_test_actual = test_data['sales'].values
y_test_baseline_pred = test_data['sales_lag_7'].values

wmape_baseline = calculate_wmape(y_test_actual, y_test_baseline_pred)
rmse_baseline = calculate_rmse(y_test_actual, y_test_baseline_pred)

print("-" * 40)
print(f"BASELINE MODEL PERFORMANCE (Last 30 Days)")
print(f"WMAPE: {wmape_baseline:.4f} ({wmape_baseline*100:.2f}%)")
print(f"RMSE:  {rmse_baseline:.4f}")
print("-" * 40)

print("Saving engineered features for LightGBM...")
df_features.to_csv('../data/processed/features_for_ml.csv', index=False)

1. Generating Seasonal Naïve Predictions (Predicting 'Lag 7' for today)...
----------------------------------------
BASELINE MODEL PERFORMANCE (Last 30 Days)
WMAPE: 1.2341 (123.41%)
RMSE:  2.9126
----------------------------------------
Saving engineered features for LightGBM...
